# Amparo -- M3 · S10 · Extra: optimizar el prompt con DSPy

Hasta aqui el prompt de generacion del RAG (`prompt_template`: instruccion +
valvula de escape) se escribio a mano. Con una metrica confiable, DSPy puede
buscar un prompt mejor: prueba instrucciones y ejemplos (few-shot), mide cada
intento y se queda con lo que mas puntua.

**Que se optimiza:** el prompt de generacion del RAG de una pasada. El retrieval
queda fijo (configuracion C, hybrid + rerank): los contextos se calculan una vez y
se reusan, asi que cualquier cambio en el puntaje es atribuible al prompt.

**Contra que metrica:** `dspy_prompt.metrica_amparo`, programatica (sin juez LLM,
para no gastar el cupo de Groq en cientos de intentos). Mide el principio de
Amparo, ser honesto con sus fuentes: en gold, responder citando solo articulos
del contexto (0.5 si responde sin citar); en adversariales, ser prudente. No mide
si la respuesta es juridicamente correcta: eso lo mide RAGAS en la evaluacion final.

**Con que datos (sin tocar el examen):**

| Conjunto | Gold | Adversariales | Uso |
|---|---|---|---|
| train | 40 del dataset de M1 | 6 nuevos | DSPy arma instrucciones y demos |
| dev | 20 del dataset de M1 | 4 nuevos | elegir entre base / BootstrapFewShot / MIPROv2 |
| test | eval set (50) | eval set (6) | una sola vez al final, comparable con las otras rutas |

**Fases:** datos y contextos → modelo (motor propio sobre HF) → base → `BootstrapFewShot` →
`MIPROv2` (light) → elegir y exportar → test con el generador de HF → RAGAS y W&B.

**Requiere GPU** (T4 o superior), el indice de `m3_s07_rag_ingenuo.ipynb` en Drive
y los secretos de Colab `GROQ_API_KEY` y `WANDB_API_KEY` (fase 9). Decisiones en
`docs/m3_decisiones_rag.md`, seccion 24.

In [1]:
# Rama del repo a clonar. Cambiala si el codigo del RAG avanzado todavia no esta
# en main (por ejemplo, mientras vive en una rama de PR).
RAMA = "m3.5"

import os

if not os.path.isdir("Amparo"):
    !git clone -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd Amparo && git fetch origin && git checkout {RAMA} && git pull

%cd Amparo

Cloning into 'Amparo'...
remote: Enumerating objects: 539, done.
remote: Counting objects: 100% (185/185), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 539 (delta 105), reused 135 (delta 75), pack-reused 354 (from 1)
Receiving objects: 100% (539/539), 2.56 MiB | 6.57 MiB/s, done.
Resolving deltas: 100% (294/294), done.
/content/Amparo


In [2]:
import os, sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

REQUERIDOS = [
    "tools/rag/hybrid.py",
    "tools/rag/rerank.py",
    "tools/rag/tools.py",
    "tools/rag/agentico.py",
    "tools/rag/dspy_prompt.py",
    "tools/rag/pipeline.py",
    "data/eval_set.json",
]
faltan = [r for r in REQUERIDOS if not os.path.exists(r)]
if faltan:
    raise SystemExit(
        f"Falta en la rama '{RAMA}': {faltan}\n\n"
        "Este notebook corre contra el repo REMOTO. Si el codigo de DSPy "
        "todavia esta solo en tu maquina, commitealo y pusheralo, o cambia RAMA."
    )
print("Repo OK.")

Repo OK.


In [3]:
# Dependencias livianas del repo (incluye faiss-cpu y rank_bm25, el BM25 de la
# hybrid search).
!pip install -q -r requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 16.8 MB/s eta 0:00:00


In [4]:
# Stack pesado: se instala aqui y no en requirements.txt, para no reemplazar el
# build de PyTorch con CUDA que Colab ya trae. sentence-transformers trae el
# cross-encoder del reranking; transformers alcanza para e5 y la generacion;
# peft/bitsandbytes solo si se genera con el adaptador LoRA.
!pip install -q -U transformers sentence-transformers peft bitsandbytes accelerate

# DSPy fijado a la version con la que se probo el codigo (optuna: lo pide MIPROv2).
!pip install -q "dspy[optuna]==3.4.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 120.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 33.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 987.5/987.5 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.2/290.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Fase 1 -- Cargar el indice y el BM25

Igual que en S08 y S10: el indice denso se copia de Drive y el BM25 se construye
una vez en memoria.

In [6]:
from google.colab import drive
import pathlib, shutil
from tools.rag import config

drive.mount("/content/drive")
origen = pathlib.Path(config.DRIVE_ROOT) / "rag"
config.ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
for nombre in ("rag_index.faiss", "rag_index_metadata.jsonl"):
    shutil.copy(origen / nombre, config.ARTIFACTS_DIR / nombre)
print("Indice copiado de Drive a artifacts/.")

Mounted at /content/drive
Indice copiado de Drive a artifacts/.


In [7]:
from tools.rag import pipeline
from tools.rag.hybrid import BM25Index

store = pipeline.load_index()
bm25 = BM25Index(store.metadata)   # una sola vez; se reusa en cada consulta
print(f"Indice denso: {len(store)} chunks | BM25: {len(bm25)} chunks")

Indice denso: 3425 chunks | BM25: 3425 chunks


## Fase 2 -- Datos y contextos

`dividir_datos` arma train y dev (determinista, semilla 42) sin ninguna pregunta
del eval set. Para cada caso se recupera el contexto con la configuracion C y se
guarda en Drive: si la sesion se cae, no hay que volver a recuperar.

In [8]:
import json, pathlib
from tools.rag import dspy_prompt as dp
from tools.rag.retrieve import retrieve
from tools.evaluation import eval_set as eval_set_mod

DSPY_DIR = pathlib.Path(config.DRIVE_ROOT) / "rag" / "dspy"
DSPY_DIR.mkdir(parents=True, exist_ok=True)
cache = DSPY_DIR / "datos_con_contexto.json"

if cache.exists():
    datos = json.loads(cache.read_text(encoding="utf-8"))
    print("Datos cargados de Drive.")
else:
    train, dev = dp.dividir_datos(dp.cargar_dataset_m1(), eval_set_mod.load_eval_set())
    def con_ctx(caso):
        return dp.con_contexto(caso, retrieve(caso["pregunta"], store, bm25=bm25, use_hybrid=True, use_rerank=True))
    datos = {"train": [con_ctx(c) for c in train], "dev": [con_ctx(c) for c in dev]}
    cache.write_text(json.dumps(datos, ensure_ascii=False, indent=1), encoding="utf-8")

for nombre, casos in datos.items():
    sin_ctx = sum(1 for c in casos if not c["articulos"])
    print(f"{nombre}: {len(casos)} casos ({sum(c['tipo'] == 'gold' for c in casos)} gold, "
          f"{sum(c['tipo'] == 'adversarial' for c in casos)} adversariales) | sin contexto: {sin_ctx}")

Datos cargados de Drive.
train: 46 casos (40 gold, 6 adversariales) | sin contexto: 7
dev: 24 casos (20 gold, 4 adversariales) | sin contexto: 5


## Fase 3 -- El modelo: motor propio sobre el generador de HF

DSPy necesita un objeto que sepa responder peticiones; no tiene que ser un
servidor externo. Se prueba con un motor propio (`dspy_prompt.crear_lm_hf`)
que llama directo al generador de HF ya cargado en la GPU -- el mismo Qwen2.5-7B
de las demas rutas de S10 -- en vez de levantar Ollama.

(Se intento primero con Ollama, que expone una API compatible con OpenAI: en
Colab resulto fragil -- instalador que necesita `zstd`, carrera de arranque
entre `ollama serve` y `ollama pull`, 404 del daemon si el modelo no queda
listo a tiempo. El motor propio evita todo eso: no hay proceso externo, ni
puerto, ni instalador.)

Por eso la optimizacion y el **test final (fase 8)** corren sobre el mismo
generador: la cifra que se reporta sale del mismo modelo que las demas rutas,
sin el matiz de cuantizacion distinta que tenia la version con Ollama.

In [9]:
from tools.rag import pipeline

# Amparo = modelo fine-tuneado de M1 (LoRA) + RAG: es el sistema que se entrega,
# asi que la corrida principal usa el adaptador. False sirve como ablacion
# ("¿el fine-tuning sigue aportando con RAG?"); los archivos de S10 quedan con
# sufijo _lora / _base, asi que las dos corridas no se pisan.
USE_LORA = True
# Se carga una sola vez aqui y se reusa en Fase 8 (mismo USE_LORA en ambas):
# asi no se duplica el modelo de 7B en la GPU, y el prompt se optimiza sobre el
# mismo modelo con el que se evalua.
model_bundle = pipeline.load_model(use_lora=USE_LORA)
print("Generador de HF cargado.")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Generador de HF cargado.


In [10]:
import dspy

lm = dp.crear_lm_hf(model_bundle, max_new_tokens=config.MAX_NEW_TOKENS_GENERATION)
dspy.configure(lm=lm)

trainset = dp.a_ejemplos_dspy(datos["train"])
devset = dp.a_ejemplos_dspy(datos["dev"])
evaluar = dspy.Evaluate(devset=devset, metric=dp.metrica_amparo, num_threads=1, display_progress=True)
print(f"LM: {lm.model} | train {len(trainset)} | dev {len(devset)}")

LM: hf/qwen2.5-7b-instruct | train 46 | dev 24


## Fase 4 -- Punto de partida: el prompt escrito a mano

Mismas reglas de `prompt_template` (instruccion + valvula de escape), sin
ejemplos, medidas con la misma metrica sobre dev. Es la vara contra la que se
compara todo lo demas.

In [11]:
puntajes, programas = {}, {}

programas["base"] = dp.construir_programa()
puntajes["base"] = evaluar(programas["base"]).score
print("Base (dev):", puntajes["base"])

  0%|          | 0/24 [00:00<?, ?it/s]

2026/09/27 22:30:34 ERROR dspy.utils.parallelizer: Error for Example({'contexto': '(No se encontro contexto relevante en la base de conocimiento.)', 'pregunta': 'Me despidieron por bajo rendimiento sin haberme llamado la atencion antes.', 'respuesta': 'En general, para un despido por esta razon suele exigirse un proceso previo que te permita conocer y corregir el supuesto incumplimiento. Reune evidencia de tu desempeno y de si recibiste algun llamado de atencion previo.', 'tipo': 'gold', 'articulos': [], 'id': 173}) (input_keys={'contexto', 'pregunta'}): LM response cannot be serialized to a JSON object.

Adapter JSONAdapter failed to parse the LM response. 

LM Response: No tengo informacion verificada sobre esto en mi base de conocimiento. Revisa tu contrato para verificar los requisitos de advertencia previa antes de un despido por bajo rendimiento. 

Expected to find output fields in the LM response: [respuesta] 

. Set `provide_traceback=True` for traceback.


Average Metric: 0.00 / 0 (0%):   4%|▍         | 1/24 [00:09<03:40,  9.61s/it]

2026/09/27 22:30:41 ERROR dspy.utils.parallelizer: Error for Example({'contexto': '(No se encontro contexto relevante en la base de conocimiento.)', 'pregunta': 'La DIAN me embargo una cuenta por una deuda de impuestos.', 'respuesta': 'Solicita copia del proceso administrativo de cobro que sustenta el embargo. Revisa el origen de la deuda y los terminos para presentar tus argumentos o solicitar facilidades de pago segun el procedimiento aplicable a este tipo de obligaciones.', 'tipo': 'gold', 'articulos': [], 'id': 485}) (input_keys={'contexto', 'pregunta'}): LM response cannot be serialized to a JSON object.

Adapter JSONAdapter failed to parse the LM response. 

LM Response: No tengo informacion verificada sobre esto en mi base de conocimiento. Revisa los documentos que te enviaron con la decision de embargo para entender los detalles especificos de tu situacion. 

Expected to find output fields in the LM response: [respuesta] 

. Set `provide_traceback=True` for traceback.


Average Metric: 5.50 / 10 (55.0%):  50%|█████     | 12/24 [01:15<01:05,  5.45s/it]

...

Average Metric: 7.00 / 12 (58.3%):  67%|██████▋   | 16/24 [01:42<00:52,  6.52s/it]

2026/09/27 22:32:16 ERROR dspy.utils.parallelizer: Error for Example({'contexto': '(No se encontro contexto relevante en la base de conocimiento.)', 'pregunta': 'Lleve mi celular a reparar y perdi toda mi informacion sin que me avisaran.', 'respuesta': 'Documenta el estado del celular antes de entregarlo, si es posible, y la orden de servicio. Presenta reclamacion al proveedor del servicio tecnico por la perdida de tu informacion, especialmente si no te advirtieron el riesgo.', 'tipo': 'gold', 'articulos': [], 'id': 431}) (input_keys={'contexto', 'pregunta'}): LM response cannot be serialized to a JSON object.

Adapter JSONAdapter failed to parse the LM response. 

LM Response: No tengo informacion verificada sobre esto en mi base de conocimiento. Revisa si la tienda tenia alguna politica o aviso sobre la seguridad de los datos antes de llevar el dispositivo. 

Expected to find output fields in the LM response: [respuesta] 

. Set `provide_traceback=True` for traceback.


Average Metric: 12.50 / 19 (65.8%): 100%|██████████| 24/24 [02:26<00:00,  6.08s/it]

2026/09/27 22:32:51 INFO dspy.evaluate.evaluate: Average Metric: 12.5 / 24 (52.1%)



Base (dev): 52.08


## Fase 5 -- BootstrapFewShot: elegir ejemplos

Corre el programa sobre train, se queda con las respuestas que sacan 1.0 en la
metrica y las usa como ejemplos (hasta 3) en el prompt. No cambia la instruccion.

In [12]:
from dspy.teleprompt import BootstrapFewShot

bfs = BootstrapFewShot(metric=dp.metrica_amparo, max_bootstrapped_demos=dp.MAX_DEMOS, max_labeled_demos=dp.MAX_DEMOS)
programas["bootstrap"] = bfs.compile(dp.construir_programa(), trainset=trainset)
puntajes["bootstrap"] = evaluar(programas["bootstrap"]).score
print("BootstrapFewShot (dev):", puntajes["bootstrap"], "| demos:", len(programas["bootstrap"].demos))

json.dump(dp.exportar_prompt(programas["bootstrap"], origen="BootstrapFewShot", puntajes=puntajes),
          open(DSPY_DIR / "prompt_bootstrap.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

 93%|█████████▎| 43/46 [04:43<00:19,  6.60s/it]


Bootstrapped 3 full traces after 43 examples for up to 1 rounds, amounting to 43 attempts.
Average Metric: 11.50 / 24 (47.9%): 100%|██████████| 24/24 [03:03<00:00,  7.66s/it]

2026/09/27 22:40:38 INFO dspy.evaluate.evaluate: Average Metric: 11.5 / 24 (47.9%)



BootstrapFewShot (dev): 47.92 | demos: 3


## Fase 6 -- MIPROv2 (light): optimizar tambien la instruccion

Propone varias instrucciones candidatas (a partir del programa, de los datos y de
los demos), las combina con conjuntos de ejemplos y busca la mejor combinacion
con optimizacion bayesiana, midiendo en dev. Es la parte que va mas alla del
ejemplo de clase. En modo `light` y con este dev, del orden de una hora en T4.

In [13]:
from dspy.teleprompt import MIPROv2

mipro = MIPROv2(metric=dp.metrica_amparo, auto="light", max_bootstrapped_demos=dp.MAX_DEMOS,
                max_labeled_demos=dp.MAX_DEMOS, num_threads=1, seed=dp.SEMILLA)
programas["mipro"] = mipro.compile(dp.construir_programa(), trainset=trainset, valset=devset, minibatch=False)
puntajes["mipro"] = evaluar(programas["mipro"]).score
print("MIPROv2 (dev):", puntajes["mipro"])

json.dump(dp.exportar_prompt(programas["mipro"], origen="MIPROv2-light", puntajes=puntajes),
          open(DSPY_DIR / "prompt_mipro.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

2026/09/27 22:40:39 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 10
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 24

2026/09/27 22:40:39 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/09/27 22:40:39 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/09/27 22:40:39 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


 93%|█████████▎| 43/46 [00:01<00:00, 28.80it/s]


Bootstrapped 3 full traces after 43 examples for up to 1 rounds, amounting to 43 attempts.
Bootstrapping set 4/6


  2%|▏         | 1/46 [00:06<04:33,  6.09s/it]


...

Average Metric: 8.00 / 14 (57.1%):  62%|██████▎   | 15/24 [00:00<00:00, 27.90it/s]

2026/09/27 22:55:18 ERROR dspy.utils.parallelizer: Error for Example({'contexto': '(No se encontro contexto relevante en la base de conocimiento.)', 'pregunta': 'Me pusieron comparendo porque llevaba el casco puesto pero sin abrochar bien.', 'respuesta': 'Revisa los detalles especificos del comparendo. Si consideras que el casco estaba correctamente puesto y hay un error en la apreciacion del agente, puedes presentar tus descargos con cualquier evidencia disponible (fotos, testigos).', 'tipo': 'gold', 'articulos': [], 'id': 526}) (input_keys={'contexto', 'pregunta'}): LM response cannot be serialized to a JSON object.

Adapter JSONAdapter failed to parse the LM response. 

LM Response: No tengo informacion verificada sobre esto en mi base de conocimiento. Revisa el comparendo para verificar las condiciones especificas de tu situacion. 

Expected to find output fields in the LM response: [respuesta] 

. Set `provide_traceback=True` for traceback.


Average Metric: 14.50 / 22 (65.9%): 100%|██████████| 24/24 [00:00<00:00, 27.84it/s]

2026/09/27 22:55:18 INFO dspy.evaluate.evaluate: Average Metric: 14.5 / 24 (60.4%)



MIPROv2 (dev): 60.42


## Fase 7 -- Elegir y exportar

Gana el mayor puntaje en dev; ante empate, el mas simple (base, luego
BootstrapFewShot). El ganador se exporta como JSON (instrucciones + demos) a
Drive. **Para versionarlo**, copien ese archivo al repo como
`results/m3_dspy_prompt.json` y comitéenlo: asi queda trazable que prompt se
evaluo.

In [14]:
orden = ["base", "bootstrap", "mipro"]
ganador = max(orden, key=lambda k: (puntajes[k], -orden.index(k)))
print("Puntajes en dev:", puntajes, "-> ganador:", ganador)

prompt_final = dp.exportar_prompt(programas[ganador], origen=ganador, puntajes=puntajes)
ruta_prompt = DSPY_DIR / "prompt_optimizado.json"
json.dump(prompt_final, open(ruta_prompt, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Guardado en:", ruta_prompt)

print("\n=== INSTRUCCIONES ===\n" + prompt_final["instrucciones"])
for i, d in enumerate(prompt_final["demos"], start=1):
    print(f"\n=== DEMO {i} ===\nPREGUNTA: {d['pregunta']}\nRESPUESTA: {d['respuesta'][:400]}")

Puntajes en dev: {'base': 52.08, 'bootstrap': 47.92, 'mipro': 60.42} -> ganador: mipro
Guardado en: /content/drive/MyDrive/Colab Notebooks/Amparo/rag/dspy/prompt_optimizado.json

=== INSTRUCCIONES ===
Eres un asistente juridico que responde consultas de derecho colombiano. Responde de forma breve, fundamentada y prudente. Cita unicamente normas que aparezcan en el CONTEXTO, y citalas exactamente como figuran ahi (ley y articulo). Si el CONTEXTO no contiene informacion suficiente para responder con fundamento, responde exactamente con la frase: "No tengo informacion verificada sobre esto en mi base de conocimiento.


## Fase 8 -- Test final sobre el eval set, con el generador de HF

Mismo `model_bundle` de la Fase 3 (ya es el generador de HF, sin Ollama que
apagar ni modelo que recargar). El prompt ganador se pasa a
`pipeline.answer_query` con `prompt_optimizado=...`; el retrieval es el mismo
(configuracion C). La salida queda marcada como `una_pasada_dspy`.

Si la corrida de S10 ya existe en Drive, se compara con su ruta `una_pasada`
(el prompt original) con la misma metrica; si no, se corre tambien el prompt
original aqui.

In [15]:
import time

USE_LORA = globals().get("USE_LORA", True)   # el de la Fase 3
etiqueta = "lora" if USE_LORA else "base"
# model_bundle ya esta cargado (Fase 3, mismo USE_LORA).
registros = eval_set_mod.load_eval_set()
corridas_dir = pathlib.Path(config.DRIVE_ROOT) / "rag" / "corridas_s10"
corridas_dir.mkdir(parents=True, exist_ok=True)

def correr(prompt=None):
    records, t0 = [], time.perf_counter()
    for reg in registros:
        r = pipeline.answer_query(reg["messages"][1]["content"], store, use_lora=USE_LORA, bm25=bm25,
                                  model_bundle=model_bundle, use_hybrid=True, use_rerank=True,
                                  prompt_optimizado=prompt)
        records.append(pipeline.to_eval_record(r, reg))
    return records, (time.perf_counter() - t0) / len(registros)

corridas, latencias = {}, {}
corridas["una_pasada_dspy"], latencias["una_pasada_dspy"] = correr(prompt_final)

archivo_base = corridas_dir / f"eval_records_una_pasada_{etiqueta}.json"
if archivo_base.exists():
    corridas["una_pasada"] = json.loads(archivo_base.read_text(encoding="utf-8"))
    lat = corridas_dir / f"latencias_{etiqueta}.json"
    latencias["una_pasada"] = json.loads(lat.read_text())["una_pasada"] if lat.exists() else None
else:
    corridas["una_pasada"], latencias["una_pasada"] = correr(None)

json.dump(corridas["una_pasada_dspy"], open(corridas_dir / f"eval_records_una_pasada_dspy_{etiqueta}.json", "w",
          encoding="utf-8"), ensure_ascii=False, indent=2)

print(f"{'ruta':<17}{'metrica test':>13}{'gold':>7}{'adv':>7}{'s/cons':>8}")
for ruta, recs in corridas.items():
    p = [dp.puntaje_de_record(r) for r in recs]
    g = [dp.puntaje_de_record(r) for r in recs if r["tipo"] == "gold"]
    a = [dp.puntaje_de_record(r) for r in recs if r["tipo"] == "adversarial"]
    lat = latencias.get(ruta)
    print(f"{ruta:<17}{sum(p)/len(p):>13.3f}{sum(g)/len(g):>7.3f}{sum(a)/len(a):>7.3f}"
          f"{(f'{lat:.1f}' if lat else '-'):>8}")

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

ruta              metrica test   gold    adv  s/cons
una_pasada_dspy          0.598  0.550  1.000     6.4
una_pasada               0.607  0.560  1.000     6.3


## Fase 9 -- RAGAS (juez Groq) y registro en W&B

La metrica de DSPy mide honestidad con las fuentes; RAGAS mide ademas si la
respuesta se apoya en el contexto y contesta lo que se pregunto. El prompt
optimizado se adopta solo si no empeora RAGAS: optimizar una metrica y
degradar las demas no es una mejora. Mismo juez, checkpoint y proyecto de W&B
que en S10.

In [19]:
import os, datetime, wandb
from google.colab import userdata
from tools.evaluation import ragas_metrics as rm, tracking, external_judge

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
juez, embed = rm.juez_groq(), rm.embedder_e5()

filas_ragas = {ruta: rm.evaluar_corrida(recs, juez=juez, embed=embed,
                                        checkpoint_path=corridas_dir / f"ragas_checkpoint_{ruta}_{etiqueta}.jsonl")
               for ruta, recs in corridas.items()}
resumenes = rm.resumen([f for fs in filas_ragas.values() for f in fs])
escapes = rm.tasas_de_escape([r for rs in corridas.values() for r in rs])
for ruta in corridas:
    r, e = resumenes.get(ruta, {}), escapes.get(ruta, {})
    print(f"{ruta:<17} faith={r.get('faithfulness')} c.prec={r.get('context_precision')} "
          f"c.rec={r.get('context_recall')} a.rel={r.get('answer_relevancy')} "
          f"prud.adv={e.get('prudencia_en_adversariales')} cita!gold={e.get('citas_no_respaldadas_en_gold')}")

try:
    wandb_key = userdata.get("WANDB_API_KEY")
except Exception:
    wandb_key = None
print("W&B en modo:", tracking.modo_de_ejecucion(wandb_key))
WANDB_ENTITY, WANDB_PROJECT = "tomasposada67-universidad-eafit", "amparo-rag"
config_run = {"generador": "Qwen/Qwen2.5-7B-Instruct", "use_lora": USE_LORA, "retrieval": "hybrid + rerank (config C)",
              "juez_ragas": external_judge.GROQ_JUDGE_MODEL, "dspy_version": dp.VERSION_DSPY,
              "optimizador_ganador": ganador, "puntajes_dev": puntajes,
              "instrucciones": prompt_final["instrucciones"][:1000], "n_demos": len(prompt_final["demos"])}
url = tracking.registrar_ruta(
    wandb, sistema="una_pasada_dspy", resumen_ragas=resumenes.get("una_pasada_dspy", {}),
    escape=escapes.get("una_pasada_dspy", {}), latencia_s=latencias.get("una_pasada_dspy"),
    filas_ragas=filas_ragas["una_pasada_dspy"], records=corridas["una_pasada_dspy"], config=config_run,
    entity=WANDB_ENTITY, project=WANDB_PROJECT, grupo=f"dspy-{etiqueta}-{datetime.date.today().isoformat()}",
)
print("W&B:", url or "(offline)")

[ragas] 10/50 -- tokens del juez en esta sesion: 5408
[ragas] limite por minuto de Groq; espero 20s y reintento...
[ragas] 15/50 -- tokens del juez en esta sesion: 14492
[ragas] 20/50 -- tokens del juez en esta sesion: 25054
[ragas] 25/50 -- tokens del juez en esta sesion: 35597
[ragas] 30/50 -- tokens del juez en esta sesion: 48884
[ragas] 35/50 -- tokens del juez en esta sesion: 61880
[ragas] 40/50 -- tokens del juez en esta sesion: 73847
[ragas] 45/50 -- tokens del juez en esta sesion: 86349
[ragas] 50/50 -- tokens del juez en esta sesion: 96535
una_pasada_dspy   faith=0.5753 c.prec=0.603 c.rec=0.51 a.rel=0.8555 prud.adv=1.0 cita!gold=0.0
una_pasada        faith=0.61 c.prec=0.5759 c.rec=0.4767 a.rel=0.8509 prud.adv=1.0 cita!gold=0.0
W&B en modo: online


answer_relevancy,▁
casos,▁
citas_no_respaldadas_en_gold,▁
context_precision,▁
context_recall,▁
escape_en_adversariales,▁
escape_en_gold,▁
faithfulness,▁
fallos_parseo,▁
latencia_s_por_consulta,▁
+9,...


W&B: https://wandb.ai/tomasposada67-universidad-eafit/amparo-rag/runs/t1k9gn6c


---
### Como leer el resultado y que hacer

- **El prompt optimizado gana** si mejora la metrica de test y no empeora RAGAS
  (sobre todo faithfulness y answer relevancy) ni la prudencia en adversariales.
  Entonces: copiar `prompt_optimizado.json` a `results/m3_dspy_prompt.json`,
  comitearlo y usarlo con `pipeline.answer_query(prompt_optimizado=dp.cargar_prompt(...))`.
- **No gana** (o gana en dev y no en test): se reporta igual. Es evidencia de que
  el prompt escrito a mano ya estaba cerca del optimo para esta metrica, o de que
  dev es chico; no se adopta.
- La latencia importa: los demos alargan el prompt de cada consulta.

*Amparo · M3 · S10 · Extra DSPy.*